# 객체 탐지 1차 학습 (공개 데이터만, ADR 0060)

Open Images V7 + LVIS v1의 가사·청소·돌봄 객체로 YOLOX-s(Apache-2.0)를 미세조정해 ONNX와 manifest를 만든다.
Kaggle·Colab 겸용. 자세한 설명은 `scripts/pretrain/README.md`.

**반드시 지킬 것**
- 이 노트북과 올린 파일(데이터셋·zip)은 **비공개(Private)** 로 둔다. Kaggle은 노트북·데이터셋을 만들 때 공개 범위를 확인한다.
- **우리 데이터(원본·블러본·라벨·골든셋·작업자 정보)는 절대 올리지 않는다.** 올리는 것은 `scripts/pretrain/`과 `config/pretrain/`뿐이다.
- 저장소 비밀값·토큰·`.env`를 넣지 않는다 (이 노트북은 비밀값이 필요 없다).
- 무료 계정의 서비스 약관(사용 목적 제한)을 쓰기 전에 확인한다.
- Kaggle: 오른쪽 설정에서 Accelerator = GPU(T4 등), Internet = On. Colab: 런타임 유형 = GPU.


## 1. 플랫폼 감지와 경로

In [ ]:
import pathlib
import shutil
import subprocess
import sys

if pathlib.Path("/kaggle").exists():
    PLATFORM = "kaggle"
elif pathlib.Path("/content").exists():
    PLATFORM = "colab"
else:
    PLATFORM = "other"
print("platform:", PLATFORM)

if PLATFORM == "kaggle":
    WORK = pathlib.Path("/kaggle/working/dlp")  # 코드 (출력으로 저장됨, 작다)
    # 이미지(수 GB)는 출력으로 저장되지 않는 곳에 둔다.
    # /kaggle/working은 버전 출력으로 저장되고 용량 제한이 있다.
    TMP = pathlib.Path("/kaggle/tmp")
    DATA = (TMP if TMP.is_dir() else pathlib.Path("/tmp")) / "objects"
    CKPT = pathlib.Path("/kaggle/working/ckpt")  # 버전 저장(Save Version) 때 출력으로 남는다
elif PLATFORM == "colab":
    WORK = pathlib.Path("/content/dlp")
    DATA = pathlib.Path("/content/objects")  # 런타임 로컬 디스크 (Drive는 느리다)
    CKPT = None  # 아래 Drive 셀에서 정한다
else:
    WORK = pathlib.Path.cwd() / "dlp"
    DATA = pathlib.Path.cwd() / "objects"
    CKPT = pathlib.Path.cwd() / "ckpt"
print(WORK, DATA, CKPT)
disks = ["/", "/tmp"] + (["/kaggle/working"] if PLATFORM == "kaggle" else [])
subprocess.run(["df", "-h", *disks], check=False)

## 2. (Colab) Google Drive 연결 — 체크포인트를 Drive에 두어 세션이 끊겨도 이어서 학습한다

In [ ]:
if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    CKPT = pathlib.Path("/content/drive/MyDrive/dlp_pretrain/ckpt")
    CKPT.mkdir(parents=True, exist_ok=True)
print("CKPT:", CKPT)

## 3. 저장소 파일 올리기

저장소는 공개가 아닐 수 있다. **`scripts/pretrain/`과 `config/pretrain/`만** 묶어 올린다 (저장소 루트에서):

```bash
zip -r pretrain_bundle.zip scripts/pretrain config/pretrain -x '*/__pycache__/*'
```

- Kaggle: 오른쪽 *Add Input → Upload* 로 `pretrain_bundle.zip`을 **비공개 데이터셋**으로 올린다 → `/kaggle/input/<이름>/` 아래에 보인다.
- Colab: 아래 셀이 업로드 창을 띄운다 (또는 Drive `MyDrive/dlp_pretrain/pretrain_bundle.zip`에 두면 그것을 쓴다).


In [ ]:
import zipfile


def find_bundle():
    kaggle_in = pathlib.Path("/kaggle/input")
    if kaggle_in.is_dir():
        hits = sorted(kaggle_in.rglob("pretrain_bundle.zip"))
        if hits:
            return hits[0]
        # Kaggle은 zip을 풀어서 보여 주기도 한다: 풀린 디렉터리를 찾는다
        hits = sorted(kaggle_in.rglob("scripts/pretrain/build_objects.py"))
        if hits:
            return hits[0].parents[2]
    for p in (
        pathlib.Path("/content/drive/MyDrive/dlp_pretrain/pretrain_bundle.zip"),
        pathlib.Path.cwd() / "pretrain_bundle.zip",
    ):
        if p.exists():
            return p
    if PLATFORM == "colab":
        from google.colab import files

        up = files.upload()
        return pathlib.Path(next(iter(up)))
    raise FileNotFoundError("pretrain_bundle.zip을 찾지 못했습니다 (3번 안내 참고)")


bundle = find_bundle()
WORK.mkdir(parents=True, exist_ok=True)
if bundle.is_dir():
    for sub in ("scripts/pretrain", "config/pretrain"):
        shutil.copytree(bundle / sub, WORK / sub, dirs_exist_ok=True)
else:
    with zipfile.ZipFile(bundle) as zf:
        names = zf.namelist()
        allowed = ("scripts/pretrain/", "config/pretrain/")
        bad = [n for n in names if not n.startswith(allowed)]
        if bad:
            raise SystemExit(f"묶음에 허용 밖의 파일이 있습니다: {bad[:5]}")
        zf.extractall(WORK)
SCRIPTS = WORK / "scripts" / "pretrain"
CONFIG = WORK / "config" / "pretrain" / "objects.yaml"
print(sorted(p.name for p in SCRIPTS.iterdir()), CONFIG.exists())

## 4. 의존성 설치 (torch는 플랫폼 것을 그대로 쓴다)

In [ ]:
import torch

subprocess.run(["nvidia-smi"], check=False)
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpu", torch.cuda.is_available())
req = SCRIPTS / "requirements.txt"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)
assert r.returncode == 0

## 5. 학습 데이터: 먼저 `--dry-run`으로 개수만 확인한다 (주석만 받음, 수 분)

In [ ]:
def run(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    r = subprocess.run([str(c) for c in cmd], cwd=WORK, check=False)
    if r.returncode != 0:
        raise SystemExit(f"실패: {r.returncode}")


build = [sys.executable, SCRIPTS / "build_objects.py", "--config", CONFIG, "--out", DATA]
run([*build, "--dry-run"])

## 6. 이미지 받기 (수십 분, 다시 실행하면 받은 것은 건너뛴다)

In [ ]:
build = [sys.executable, SCRIPTS / "build_objects.py", "--config", CONFIG, "--out", DATA]
run([*build, "--workers", "16"])
subprocess.run(["du", "-sh", str(DATA)], check=False)

## 7. 학습 (`--resume`: 체크포인트가 있으면 이어서, 없으면 YOLOX-s COCO 가중치에서 시작)

첫 epoch 로그의 `ETA`를 보고 세션 시간(Kaggle 12시간, Colab 무료는 더 짧고 일정하지 않음) 안에 끝나지 않으면
`EPOCHS`를 줄이거나, 끊긴 뒤 이 셀부터 다시 실행해 이어서 한다.
Kaggle에서 이전 버전의 `ckpt`를 이어 쓰려면 그 버전 출력을 입력으로 붙이고 `RESUME_FROM`에 경로를 적는다.


In [ ]:
EPOCHS = None  # None이면 objects.yaml train.epochs
RESUME_FROM = None  # 예: "/kaggle/input/<이전 버전 출력>/ckpt"
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, CKPT, dirs_exist_ok=True)
cmd = [
    sys.executable,
    SCRIPTS / "train_yolox.py",
    "--config",
    CONFIG,
    "--data",
    DATA,
    "--ckpt-dir",
    CKPT,
    "--work",
    WORK / "work",
    "--resume",
    "--workers",
    "2",
]
if EPOCHS:
    cmd += ["--epochs", str(EPOCHS)]
run(cmd)

## 8. 결과 저장 (ONNX + model_manifest.json + data_manifest.json + pip_freeze.txt)

In [ ]:
export = CKPT / "export"
shutil.copy2(DATA / "data_manifest.json", export / "data_manifest.json")
print(sorted(p.name for p in export.iterdir()))
if PLATFORM == "kaggle":
    print("Kaggle: Save Version 뒤 Output 탭에서 ckpt/export/를 내려받는다 (비공개 유지).")
elif PLATFORM == "colab":
    print("Colab: Drive", export, "에 있다.")